In [3]:
from pydantic import BaseModel, Field
from typing import Literal
from datetime import datetime


In [4]:
class Candidate(BaseModel):
  id: str
  title: str
  source: Literal['arxiv', 'github', 'hn']
  url: str
  summary: str
  novelty_score: float = Field(ge=0, le=1)
  discovered_at: datetime = Field(default_factory=datetime.utcnow)

In [5]:
class Milestone(BaseModel):
  name: str
  objective:str
  depends_on: list[str] = []

class Blueprint(BaseModel):
  candidate_id: str
  modules: list[str]
  milestones: list[str]
  estimated_sessions: int

class EvalResult(BaseModel):
  candidate_id: str
  passed: bool
  notes: str

In [7]:
import yaml

PROFILE = {
    "os": "Windows 11",
    "cpu": "Intel i5",
    "ram_gb": 8,
    "gpu": None,
    "preferred_stack": ["Python", "LangGraph", "SQLite", "Groq API"],
    "avoid": [
        "Docker-heavy pipelines",
        "local GPU-required models",
        "Neo4j"
    ]
}

with open("hardware_profile.yaml", "w") as f:
    yaml.dump(PROFILE, f)

def load_profile(path="hardware_profile.yaml") -> dict:
    with open(path) as f:
        return yaml.safe_load(f)

print(load_profile())

{'avoid': ['Docker-heavy pipelines', 'local GPU-required models', 'Neo4j'], 'cpu': 'Intel i5', 'gpu': None, 'os': 'Windows 11', 'preferred_stack': ['Python', 'LangGraph', 'SQLite', 'Groq API'], 'ram_gb': 8}


SQLite store

In [8]:
import sqlite3
import json

DB_PATH = 'parxis.db'

def init_db():
  conn = sqlite3.connect(DB_PATH)
  conn.execute(
    """
    CREATE TABLE IF NOT EXISTS candidates (
            id TEXT PRIMARY KEY,
            title TEXT,
            source TEXT,
            url TEXT,
            summary TEXT,
            novelty_score REAL,
            discovered_at TEXT
        )
    """
  )
  conn.execute("""
        CREATE TABLE IF NOT EXISTS blueprints (
            candidate_id TEXT PRIMARY KEY,
            modules TEXT,
            milestones TEXT,
            estimated_sessions INTEGER
        )
    """)

  conn.commit()
  conn.close()

def save_candidate(c: Candidate):
  conn = sqlite3.connect(DB_PATH)
  conn.execute(
    "INSERT OR REPLACE INTO candidates VALUES (?, ?, ?, ?, ?, ?, ?)",
    (c.id, c.title, c.source, c.url, c.summary, c.novelty_score, c.discovered_at.isoformat())
  )
  conn.commit()
  conn.close()

def get_all_candidates()->list[dict]:
  conn= sqlite3.connect(DB_PATH)
  conn.row_factory = sqlite3.Row
  rows = conn.execute("SELECT * FROM candidates").fetchall()
  conn.close()
  return [dict(r) for r in rows]

init_db()
